In [0]:
%run ../00_common/calc_ctable

In [0]:
# 使用双层解析处理嵌套 JSON 字符串 
def parse_json_to_array_smart(col):
    """
    智能解析 JSON 字符串为 Array[Map]:
    - 如果 col 是 '[{...}]'（JSON 数组）→ 直接解析为 ArrayType(MapType)
    - 如果 col 是 '{...}'（JSON 对象）→ 解析为 Map，再包成 Array
    - 如果是 null → 返回 null
    """
    map_schema = MapType(StringType(), StringType())
    array_schema = ArrayType(map_schema)

    is_array = col.startswith("[")

    parsed_as_array = F.from_json(col, array_schema)
    parsed_as_map_then_wrap = F.array(F.from_json(col, map_schema))

    return (
        F.when(col.isNull(), F.lit(None))
        .when(is_array, parsed_as_array)
        .otherwise(parsed_as_map_then_wrap)
    )

In [0]:
def calc_t_touchpoint_master_list(table_name):
    
    # === Step 2: 读表 ===
    delta_table = f"{get_env_config('golden_touchpoint_master_database')}.t_touchpoint_master_sap"
    df = spark.table(delta_table) \
        .filter(F.isnotnull("MarketCode") & F.isnotnull("BrandCode") & F.isnotnull("TouchPointCode"))

    # 应用到五个字段（注意传入正确的内部字段名）
    df_parsed = df \
        .withColumn("Attr_CustomAttributeList", parse_json_to_array_smart(F.col("Attr_CustomAttributeList"))) \
        .withColumn("TP_CustomAttributeList",   parse_json_to_array_smart(F.col("CustomAttributeList"))) \
        .withColumn("PhoneList",                parse_json_to_array_smart(F.col("PhoneList"))) \
        .withColumn("AddressList",              parse_json_to_array_smart(F.col("AddressList"))) \
        .withColumn("TerminalRegistrationList", parse_json_to_array_smart(F.col("TerminalRegistrationList")))

    # === Step 4: 构建嵌套结构（保持不变）===

    header = F.struct(
        F.col("ACTION").alias("@Action"),
        F.col("DOCUMENTTIMESTAMP").alias("DocumentTimestamp"),
        F.col("DOCUMENTUUID").alias("DocumentUUID")
    )

    source_system = F.struct(
        F.col("SourceSystemCode").alias("@Code"),
        F.col("SourceTimestamp").alias("SourceTimestamp"),
        F.col("MarketCode").alias("MarketCode"),
        F.col("AffiliateCode").alias("AffiliateCode"),
        F.col("DivisionCode").alias("DivisionCode"),
        F.col("BrandCode").alias("BrandCode"),
        F.col("TouchPointCode").alias("TouchPointCode"),
        F.col("SubMarketCode").alias("SubMarketCode")
    )

    auxiliary_source_system = F.when(
        F.col("AuxiliaryCode").isNull() & F.col("AuxiliaryTouchPointCode").isNull(),
        F.lit(None)
    ).otherwise(
        F.struct(
            F.col("AuxiliaryCode").alias("@Code"),
            F.col("AuxiliaryTouchPointCode").alias("TouchPointCode")
        )
    )

    hygiene = F.when(
        F.col("HygieneServiceCode").isNull(),
        F.lit(None)
    ).otherwise(
        F.struct(
            F.col("HygieneServiceCode").alias("HygieneServiceCode")
        )
    )

    org_hierarchy_array_all = F.array(
        F.struct(
            F.lit("Global").alias("Level"),
            F.col("Global_Code").alias("Code"),
            F.col("Global_Name").alias("Name"),
            F.col("Global_Description").alias("Description")
        ),
        F.struct(
            F.lit("Regional").alias("Level"),
            F.col("Regional_Code").alias("Code"),
            F.col("Regional_Name").alias("Name"),
            F.col("Regional_Description").alias("Description")
        ),
        F.struct( 
            F.lit("Affiliate").alias("Level"),
            F.col("Affiliate_Code").alias("Code"),
            F.col("Affiliate_Name").alias("Name"),
            F.col("Affiliate_Description").alias("Description")
        )
    )
    
    org_hierarchy_array = F.filter(
        org_hierarchy_array_all,
        lambda x: x["Code"].isNotNull()
    )

    organization_hierarchy_list = F.struct(
        org_hierarchy_array.alias("OrganizationHierarchy")
    )

    contact_information = F.struct(
        F.when(
            F.col("PhoneList").isNull(),
            F.lit(None)
        ).otherwise(
            F.struct(F.col("PhoneList").alias("Phone"))
        ).alias("PhoneList"),
        F.when(
            F.col("AddressList").isNull(),
            F.lit(None)
        ).otherwise(
            F.struct(F.col("AddressList").alias("Address"))
        ).alias("AddressList")
    )

    attributes = F.struct(
        F.col("DistributionChannelCode"),
        F.col("TouchPointGroupCode"),
        F.col("RetailerHierarchyCode"),
        F.col("TouchPointTypeCode"),
        F.col("EnglishDescription"),
        F.col("LocalDescription"),
        F.col("EnglishFullDescription"),
        F.col("LocalFullDescription"),
        F.col("Descriptionen").alias("Description_en"),
        F.col("Descriptionlocal").alias("Description_local"),
        F.col("FullDescriptionen").alias("FullDescription_en"),
        F.col("FullDescriptionlocal").alias("FullDescription_local"),
        F.col("URL"),
        F.col("JDECode"),
        F.col("Active"),
        F.col("TouchPointStatus"),
        F.col("OpenDate"), 
        F.col("BranchID"),
        F.col("RedirectTouchPointCode"),
        F.col("Channel"),
        F.col("CustomerGroup"),
        F.col("DTCFlag"),
        F.col("Region"),
        F.col("City"),
        F.when(
            F.col("Attr_CustomAttributeList").isNull(),
            F.lit(None)
        ).otherwise(
            F.struct(F.col("Attr_CustomAttributeList").alias("CustomAttribute"))
        ).alias("CustomAttributeList")
    )

    touchpoint = F.struct(
        F.col("RECORDUUID").alias("@RecordUUID"),
        source_system.alias("SourceSystem"),
        auxiliary_source_system.alias("AuxiliarySourceSystem"),
        hygiene.alias("Hygiene"),
        attributes.alias("Attributes"),
        contact_information.alias("ContactInformation"),
        organization_hierarchy_list.alias("OrganizationHierarchyList"),
        F.when(
            F.col("TP_CustomAttributeList").isNull(),
            F.lit(None)
        ).otherwise(
            F.struct(F.col("TP_CustomAttributeList").alias("CustomAttribute"))
        ).alias("CustomAttributeList"),
        F.struct(
            F.when(
                F.col("TerminalRegistrationList").isNull(),
                F.expr("array()").cast("array<map<string,string>>")
            ).otherwise(F.col("TerminalRegistrationList")).alias("TerminalRegistration")
        ).alias("TerminalRegistrationList"),
        F.col("CustomerNumber").alias("CustomerNumber")
    )

    reconstructed_payload = F.struct(
        header.alias("Header"),
        touchpoint.alias("TouchPoint")
    )

    final_df = df_parsed.select(
        F.col("DivisionCode"),
        F.col("CustomerNumber"),
        F.col("MarketCode"),
        F.col("BrandCode"),
        F.col("TouchPointCode"),
        F.to_json(reconstructed_payload, options={"ignoreNullFields": "false"}).alias("TouchPointMasterJSON"),
        F.col("SourceTimestamp"),
        F.col("TP_SAPBI_ID"),
        F.col("TCPT_ID"),
        F.col("KAFKA_TIMESTAMP")
    )

    #如果需要写回，取消注释以下行
    merge_condition = get_merge_condition(["MarketCode", "BrandCode", "TouchPointCode"])
    update_condition = get_update_condition(["DivisionCode", "CustomerNumber", "TouchPointMasterJSON", "SourceTimestamp"]) #"KAFKA_TIMESTAMP"
    merge_t_table(table_name, final_df, merge_condition, update_condition)
    # calc ctable
    calc_ctable(table_name, None)

In [0]:
batch_id = dbutils.widgets.get("batch_id")
print(f"batch_id: {batch_id}")

table_name = f"{get_env_config('golden_touchpoint_combine_database')}.t_touchpoint_dataset"
print(f"table_name: {table_name}")

with StepLogger("t_touchpoint_dataset", "07", "touchpoint", task_id=batch_id) as logger:
    calc_t_touchpoint_master_list(table_name)